In [14]:
# ============================================================
# 09 - 2025-2030 TOPSIS ÖNEM / İHTİYAÇ SIRALAMASI
# ============================================================

import os
import re
import numpy as np
import pandas as pd


# ============================================================
# 0) DOSYA YOLLARI
# ============================================================

TOPSIS_VERI_PATH = "2025_2030_tum_hedefler_final_tahmin.csv"
ENSEMBLE_PATH = "tahmin_2025_2030_ensemble_sabit.csv"

GELECEK_YILLAR = list(range(2025, 2031))




In [16]:
# ============================================================
# 1) TOPSIS AĞIRLIKLARI VE KRİTER TİPLERİ
# ============================================================

WEIGHTS = {
    "oran_65_plus": 0.35,
    "oran_0_14": 0.10,
    "hekim_1000": 0.35,
    "yatak_100000": 0.20,
}

CRITERIA_TYPES = {
    "oran_65_plus": "benefit",
    "oran_0_14": "benefit",
    "hekim_1000": "cost",
    "yatak_100000": "cost",
}

CRITERIA_COLS = list(WEIGHTS.keys())




In [18]:
# ============================================================
# 2) YARDIMCI FONKSİYONLAR
# ============================================================

TR_MAP = str.maketrans({
    "İ": "i", "I": "i", "ı": "i",
    "Ş": "s", "ş": "s",
    "Ğ": "g", "ğ": "g",
    "Ü": "u", "ü": "u",
    "Ö": "o", "ö": "o",
    "Ç": "c", "ç": "c",
})

def normalize_city_name(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    x = re.sub(r"-\d+$", "", x)
    x = x.translate(TR_MAP).lower()
    x = re.sub(r"[^a-z0-9]+", "_", x)
    x = re.sub(r"_+", "_", x).strip("_")
    return x


def assign_priority_group(rank):
    """
    81 il için yaklaşık öncelik grubu.
    1-20   : Çok yüksek öncelik
    21-40  : Yüksek öncelik
    41-60  : Orta öncelik
    61-81  : Düşük öncelik
    """
    if rank <= 20:
        return "Çok yüksek öncelik"
    elif rank <= 40:
        return "Yüksek öncelik"
    elif rank <= 60:
        return "Orta öncelik"
    else:
        return "Düşük öncelik"




In [20]:
# ============================================================
# 3) VERİLERİ OKU
# ============================================================

topsis_raw = pd.read_csv(TOPSIS_VERI_PATH)
ensemble = pd.read_csv(ENSEMBLE_PATH)

topsis_raw.columns = [c.strip() for c in topsis_raw.columns]
ensemble.columns = [c.strip() for c in ensemble.columns]

topsis_raw["yıl"] = pd.to_numeric(topsis_raw["yıl"], errors="coerce").astype(int)
ensemble["yıl"] = pd.to_numeric(ensemble["yıl"], errors="coerce").astype(int)

if "il_key" not in topsis_raw.columns:
    topsis_raw["il_key"] = topsis_raw["il_adı"].apply(normalize_city_name)

if "il_key" not in ensemble.columns:
    ensemble["il_key"] = ensemble["il_adı"].apply(normalize_city_name)

topsis_raw = topsis_raw[topsis_raw["yıl"].isin(GELECEK_YILLAR)].copy()
ensemble = ensemble[ensemble["yıl"].isin(GELECEK_YILLAR)].copy()

print("TOPSIS veri:", topsis_raw.shape)
print("Ensemble veri:", ensemble.shape)

display(topsis_raw.head())
display(ensemble.head())




TOPSIS veri: (486, 14)
Ensemble veri: (486, 10)


,il_adı,il_key,yıl,nufus,nufus_veri_turu,nufus_kaynak,yas_0_14,yas_65_plus,hekim_oran,yatak_oran,yas_0_14_tip,yas_65_plus_tip,hekim_oran_tip,yatak_oran_tip
0,Adana,adana,2025,2283609,Gerçek,TÜİK ADNKS 2025,21.993213,10.586751,2.602272,323.985432,actual,actual,predicted,predicted
1,Adana,adana,2026,2293499,Güncel Tahmin,TÜİK Ana Senaryo + Bias-Correction,21.286629,11.016234,2.631855,329.220907,predicted,predicted,predicted,predicted
2,Adana,adana,2027,2299727,Güncel Tahmin,TÜİK Ana Senaryo + Bias-Correction,20.825061,11.457670,2.657828,334.550691,predicted,predicted,predicted,predicted
3,Adana,adana,2028,2305551,Güncel Tahmin,TÜİK Ana Senaryo + Bias-Correction,20.374964,11.919366,2.694466,339.729930,predicted,predicted,predicted,predicted
4,Adana,adana,2029,2310973,Güncel Tahmin,TÜİK Ana Senaryo + Bias-Correction,19.750900,12.396739,2.736625,344.715297,predicted,predicted,predicted,predicted


,il_adı,yıl,nufus,doktor,ebe,diger,hemsire,yatak,hastane,il_key
0,Adana,2025,2283609.0,5948.0,1735.0,8415.0,6921.0,7460.0,32.0,adana
1,Adana,2026,2293499.0,6082.0,1749.0,8949.0,6921.0,7528.0,33.0,adana
2,Adana,2027,2299727.0,6217.0,1762.0,9479.0,6921.0,7596.0,33.0,adana
3,Adana,2028,2305551.0,6352.0,1776.0,10007.0,6921.0,7663.0,33.0,adana
4,Adana,2029,2310973.0,6487.0,1790.0,10531.0,6921.0,7730.0,34.0,adana


In [22]:
# ============================================================
# 4) TOPSIS KRİTERLERİNİ HAZIRLA
# ============================================================

topsis_criteria = topsis_raw.copy()

# Kolonları TOPSIS isimlerine çevir
rename_map = {
    "yas_65_plus": "oran_65_plus",
    "yas_0_14": "oran_0_14",
    "hekim_oran": "hekim_1000",
    "yatak_oran": "yatak_100000",
}

topsis_criteria = topsis_criteria.rename(columns=rename_map)

required_topsis_cols = [
    "il_adı", "il_key", "yıl", "nufus",
    "oran_65_plus", "oran_0_14", "hekim_1000", "yatak_100000"
]

missing_cols = [c for c in required_topsis_cols if c not in topsis_criteria.columns]

if missing_cols:
    raise ValueError(f"TOPSIS kriter veri setinde eksik kolonlar var: {missing_cols}")

topsis_criteria = topsis_criteria[required_topsis_cols].copy()

# Numeric dönüşüm
for col in ["nufus"] + CRITERIA_COLS:
    topsis_criteria[col] = pd.to_numeric(topsis_criteria[col], errors="coerce")

print("TOPSIS kriter matrisi:")
display(topsis_criteria.head())




TOPSIS kriter matrisi:


,il_adı,il_key,yıl,nufus,oran_65_plus,oran_0_14,hekim_1000,yatak_100000
0,Adana,adana,2025,2283609,10.586751,21.993213,2.602272,323.985432
1,Adana,adana,2026,2293499,11.016234,21.286629,2.631855,329.220907
2,Adana,adana,2027,2299727,11.457670,20.825061,2.657828,334.550691
3,Adana,adana,2028,2305551,11.919366,20.374964,2.694466,339.729930
4,Adana,adana,2029,2310973,12.396739,19.750900,2.736625,344.715297


In [24]:
# ============================================================
# 5) ENSEMBLE TAHMİNLERİ İLE BİRLEŞTİR
# ============================================================

ensemble_cols = [
    "il_key", "yıl",
    "doktor", "ebe", "diger", "hemsire", "yatak", "hastane"
]

available_ensemble_cols = [c for c in ensemble_cols if c in ensemble.columns]

future = topsis_criteria.merge(
    ensemble[available_ensemble_cols],
    on=["il_key", "yıl"],
    how="left"
)

print("Birleşmiş TOPSIS + ensemble veri:", future.shape)
display(future.head())




Birleşmiş TOPSIS + ensemble veri: (486, 14)


,il_adı,il_key,yıl,nufus,oran_65_plus,oran_0_14,hekim_1000,yatak_100000,doktor,ebe,diger,hemsire,yatak,hastane
0,Adana,adana,2025,2283609,10.586751,21.993213,2.602272,323.985432,5948.0,1735.0,8415.0,6921.0,7460.0,32.0
1,Adana,adana,2026,2293499,11.016234,21.286629,2.631855,329.220907,6082.0,1749.0,8949.0,6921.0,7528.0,33.0
2,Adana,adana,2027,2299727,11.457670,20.825061,2.657828,334.550691,6217.0,1762.0,9479.0,6921.0,7596.0,33.0
3,Adana,adana,2028,2305551,11.919366,20.374964,2.694466,339.729930,6352.0,1776.0,10007.0,6921.0,7663.0,33.0
4,Adana,adana,2029,2310973,12.396739,19.750900,2.736625,344.715297,6487.0,1790.0,10531.0,6921.0,7730.0,34.0


In [26]:
# ============================================================
# 6) AĞIRLIKLI HAM KRİTERLERİ HESAPLA
# Bu kısım sadece yorumlama içindir.
# Asıl TOPSIS skorunda normalize edilmiş ağırlıklı değerler kullanılır.
# ============================================================

for col, w in WEIGHTS.items():
    future[f"{col}_agirlikli_ham"] = future[col] * w

print("Ham değer × ağırlık sonuçları:")
display(
    future[
        [
            "il_adı", "yıl",
            "oran_65_plus", "oran_65_plus_agirlikli_ham",
            "oran_0_14", "oran_0_14_agirlikli_ham",
            "hekim_1000", "hekim_1000_agirlikli_ham",
            "yatak_100000", "yatak_100000_agirlikli_ham",
        ]
    ].head(20).round(4)
)




Ham değer × ağırlık sonuçları:


,il_adı,yıl,oran_65_plus,oran_65_plus_agirlikli_ham,oran_0_14,oran_0_14_agirlikli_ham,hekim_1000,hekim_1000_agirlikli_ham,yatak_100000,yatak_100000_agirlikli_ham
0,Adana,2025,10.5868,3.7054,21.9932,2.1993,2.6023,0.9108,323.9854,64.7971
1,Adana,2026,11.0162,3.8557,21.2866,2.1287,2.6319,0.9211,329.2209,65.8442
2,Adana,2027,11.4577,4.0102,20.8251,2.0825,2.6578,0.9302,334.5507,66.9101
3,Adana,2028,11.9194,4.1718,20.3750,2.0375,2.6945,0.9431,339.7299,67.9460
4,Adana,2029,12.3967,4.3389,19.7509,1.9751,2.7366,0.9578,344.7153,68.9431
5,Adana,2030,12.8892,4.5112,18.9249,1.8925,2.7835,0.9742,349.5566,69.9113
6,Adıyaman,2025,9.2082,3.2229,26.0749,2.6075,1.6944,0.5930,188.8448,37.7690
7,Adıyaman,2026,9.5785,3.3525,25.3445,2.5344,1.7590,0.6156,195.1630,39.0326
8,Adıyaman,2027,9.9684,3.4889,24.6222,2.4622,1.8306,0.6407,201.2098,40.2420
9,Adıyaman,2028,10.3712,3.6299,24.0260,2.4026,1.8941,0.6630,207.2134,41.4427


In [28]:
# ============================================================
# 7) TOPSIS FONKSİYONU
# ============================================================

def calculate_topsis_for_one_year(df_year, criteria_cols, weights, criteria_types):
    """
    Tek bir yıl için iller arasında TOPSIS skoru hesaplar.
    """

    d = df_year.copy().reset_index(drop=True)

    # Eksik değerleri yıl medyanı ile doldur
    for col in criteria_cols:
        d[col] = pd.to_numeric(d[col], errors="coerce")
        d[col] = d[col].fillna(d[col].median())

    X = d[criteria_cols].astype(float).values

    # 1) Vektör normalizasyonu
    denom = np.sqrt((X ** 2).sum(axis=0))
    denom = np.where(denom == 0, 1e-12, denom)

    X_norm = X / denom

    # 2) Ağırlıklandırma
    weight_vector = np.array([weights[col] for col in criteria_cols], dtype=float)
    X_weighted = X_norm * weight_vector

    # Normalize edilmiş ve ağırlıklandırılmış değerleri tabloya ekle
    for idx, col in enumerate(criteria_cols):
        d[f"{col}_norm"] = X_norm[:, idx]
        d[f"{col}_agirlikli"] = X_weighted[:, idx]

    # 3) İdeal iyi ve ideal kötü değerler
    ideal_best = []
    ideal_worst = []

    for idx, col in enumerate(criteria_cols):
        if criteria_types[col] == "benefit":
            ideal_best.append(X_weighted[:, idx].max())
            ideal_worst.append(X_weighted[:, idx].min())
        elif criteria_types[col] == "cost":
            ideal_best.append(X_weighted[:, idx].min())
            ideal_worst.append(X_weighted[:, idx].max())
        else:
            raise ValueError(f"Bilinmeyen kriter tipi: {col} = {criteria_types[col]}")

    ideal_best = np.array(ideal_best)
    ideal_worst = np.array(ideal_worst)

    # 4) İdeal iyi ve kötüye uzaklık
    d_plus = np.sqrt(((X_weighted - ideal_best) ** 2).sum(axis=1))
    d_minus = np.sqrt(((X_weighted - ideal_worst) ** 2).sum(axis=1))

    # 5) TOPSIS skoru
    topsis_score = d_minus / (d_plus + d_minus + 1e-12)

    d["topsis_skor"] = topsis_score
    d["ideal_iyi_uzaklik"] = d_plus
    d["ideal_kotu_uzaklik"] = d_minus

    # 6) Sıralama
    d["topsis_sira"] = d["topsis_skor"].rank(
        ascending=False,
        method="dense"
    ).astype(int)

    d["oncelik_grubu"] = d["topsis_sira"].apply(assign_priority_group)

    return d




In [30]:
# ============================================================
# 8) HER YIL İÇİN TOPSIS HESAPLA
# ============================================================

topsis_results = []

for year in sorted(future["yıl"].dropna().unique()):
    df_year = future[future["yıl"] == year].copy()

    result_year = calculate_topsis_for_one_year(
        df_year=df_year,
        criteria_cols=CRITERIA_COLS,
        weights=WEIGHTS,
        criteria_types=CRITERIA_TYPES
    )

    topsis_results.append(result_year)

topsis_all = pd.concat(topsis_results, ignore_index=True)

topsis_all = topsis_all.sort_values(
    ["yıl", "topsis_sira"]
).reset_index(drop=True)

print("TOPSIS sonuçları:", topsis_all.shape)

display(
    topsis_all[
        [
            "yıl", "topsis_sira", "il_adı", "topsis_skor", "oncelik_grubu",
            "oran_65_plus", "oran_0_14", "hekim_1000", "yatak_100000",
            "oran_65_plus_agirlikli",
            "oran_0_14_agirlikli",
            "hekim_1000_agirlikli",
            "yatak_100000_agirlikli",
        ]
    ].head(30).round(5)
)




TOPSIS sonuçları: (486, 31)


,yıl,topsis_sira,il_adı,topsis_skor,oncelik_grubu,oran_65_plus,oran_0_14,hekim_1000,yatak_100000,oran_65_plus_agirlikli,oran_0_14_agirlikli,hekim_1000_agirlikli,yatak_100000_agirlikli
0,2025,1,Sinop,0.81134,Çok yüksek öncelik,21.65527,14.66384,1.58250,299.70587,0.06250,0.00788,0.02752,0.02114
1,2025,2,Artvin,0.79218,Çok yüksek öncelik,19.24897,14.68624,1.81895,226.76135,0.05555,0.00789,0.03163,0.01599
2,2025,3,Çankırı,0.78198,Çok yüksek öncelik,18.17661,16.05493,1.75383,238.95928,0.05246,0.00862,0.03050,0.01685
3,2025,4,Kastamonu,0.77305,Çok yüksek öncelik,21.08498,14.38355,1.86808,318.52948,0.06085,0.00772,0.03249,0.02246
4,2025,5,Bartın,0.76567,Çok yüksek öncelik,17.40176,14.28267,1.67340,251.48706,0.05022,0.00767,0.02910,0.01774
5,2025,6,Amasya,0.74663,Çok yüksek öncelik,17.17790,16.09241,1.77606,280.97725,0.04958,0.00864,0.03089,0.01982
6,2025,7,Burdur,0.73761,Çok yüksek öncelik,17.42585,15.21322,1.86837,286.91262,0.05029,0.00817,0.03249,0.02023
7,2025,8,Gümüşhane,0.73723,Çok yüksek öncelik,16.15769,15.79243,1.77635,236.39108,0.04663,0.00848,0.03089,0.01667
8,2025,9,Kırklareli,0.73409,Çok yüksek öncelik,17.03790,14.57290,1.86235,271.63280,0.04917,0.00783,0.03239,0.01916
9,2025,10,Balıkesir,0.72941,Çok yüksek öncelik,18.16698,15.22347,2.08055,275.08535,0.05243,0.00818,0.03618,0.01940


In [32]:
# ============================================================
# 9) ÖNEM PAYI / İHTİYAÇ PAYI OLUŞTUR
# ============================================================

topsis_all["ihtiyac_payi"] = topsis_all["topsis_skor"]

topsis_all["ihtiyac_payi_norm"] = (
    topsis_all.groupby("yıl")["ihtiyac_payi"]
    .transform(lambda x: x / x.sum() if x.sum() != 0 else 1 / len(x))
)

print("Yıllara göre ihtiyaç payı toplamı:")
display(
    topsis_all.groupby("yıl")["ihtiyac_payi_norm"]
    .sum()
    .reset_index()
)




Yıllara göre ihtiyaç payı toplamı:


,yıl,ihtiyac_payi_norm
0,2025,1.0
1,2026,1.0
2,2027,1.0
3,2028,1.0
4,2029,1.0
5,2030,1.0


In [34]:
# ============================================================
# 10) ÇIKTI TABLOLARI
# ============================================================

score_cols = [
    "il_adı", "il_key", "yıl", "nufus",
    "oran_65_plus", "oran_0_14", "hekim_1000", "yatak_100000",

    # Ham ağırlık çarpımları
    "oran_65_plus_agirlikli_ham",
    "oran_0_14_agirlikli_ham",
    "hekim_1000_agirlikli_ham",
    "yatak_100000_agirlikli_ham",

    # TOPSIS normalizasyon sonrası ağırlıklı değerler
    "oran_65_plus_norm",
    "oran_0_14_norm",
    "hekim_1000_norm",
    "yatak_100000_norm",

    "oran_65_plus_agirlikli",
    "oran_0_14_agirlikli",
    "hekim_1000_agirlikli",
    "yatak_100000_agirlikli",

    # TOPSIS sonuçları
    "ideal_iyi_uzaklik",
    "ideal_kotu_uzaklik",
    "topsis_skor",
    "topsis_sira",
    "oncelik_grubu",
    "ihtiyac_payi_norm",
]

ensemble_output_cols = [
    "doktor", "ebe", "diger", "hemsire", "yatak", "hastane"
]

for col in ensemble_output_cols:
    if col in topsis_all.columns:
        score_cols.append(col)

scores_out = topsis_all[score_cols].copy()
scores_out = scores_out.sort_values(["yıl", "topsis_sira"]).reset_index(drop=True)

weighted_cols = [
    "il_adı", "il_key", "yıl",
    "oran_65_plus", "oran_65_plus_agirlikli_ham", "oran_65_plus_agirlikli",
    "oran_0_14", "oran_0_14_agirlikli_ham", "oran_0_14_agirlikli",
    "hekim_1000", "hekim_1000_agirlikli_ham", "hekim_1000_agirlikli",
    "yatak_100000", "yatak_100000_agirlikli_ham", "yatak_100000_agirlikli",
    "topsis_skor", "topsis_sira", "oncelik_grubu"
]

weighted_out = topsis_all[weighted_cols].copy()
weighted_out = weighted_out.sort_values(["yıl", "topsis_sira"]).reset_index(drop=True)

display(scores_out.head(30).round(5))

,il_adı,il_key,yıl,nufus,oran_65_plus,oran_0_14,hekim_1000,yatak_100000,oran_65_plus_agirlikli_ham,oran_0_14_agirlikli_ham,...,topsis_skor,topsis_sira,oncelik_grubu,ihtiyac_payi_norm,doktor,ebe,diger,hemsire,yatak,hastane
0,Sinop,sinop,2025,225848,21.65527,14.66384,1.58250,299.70587,7.57935,1.46638,...,0.81134,1,Çok yüksek öncelik,0.01678,337.0,234.0,1010.0,831.0,669.0,7.0
1,Artvin,artvin,2025,167531,19.24897,14.68624,1.81895,226.76135,6.73714,1.46862,...,0.79218,2,Çok yüksek öncelik,0.01638,285.0,205.0,730.0,473.0,367.0,8.0
2,Çankırı,cankiri,2025,200549,18.17661,16.05493,1.75383,238.95928,6.36181,1.60549,...,0.78198,3,Çok yüksek öncelik,0.01617,327.0,164.0,889.0,472.0,470.0,9.0
3,Kastamonu,kastamonu,2025,379934,21.08498,14.38355,1.86808,318.52948,7.37974,1.43836,...,0.77305,4,Çok yüksek öncelik,0.01599,675.0,306.0,1562.0,1169.0,1176.0,21.0
4,Bartın,bartin,2025,206663,17.40176,14.28267,1.67340,251.48706,6.09062,1.42827,...,0.76567,5,Çok yüksek öncelik,0.01584,340.0,191.0,717.0,579.0,510.0,4.0
5,Amasya,amasya,2025,342242,17.17790,16.09241,1.77606,280.97725,6.01227,1.60924,...,0.74663,6,Çok yüksek öncelik,0.01544,615.0,339.0,1543.0,1066.0,947.0,8.0
6,Burdur,burdur,2025,277226,17.42585,15.21322,1.86837,286.91262,6.09905,1.52132,...,0.73761,7,Çok yüksek öncelik,0.01526,511.0,333.0,1237.0,913.0,774.0,8.0
7,Gümüşhane,gumushane,2025,138807,16.15769,15.79243,1.77635,236.39108,5.65519,1.57924,...,0.73723,8,Çok yüksek öncelik,0.01525,250.0,114.0,611.0,450.0,353.0,6.0
8,Kırklareli,kirklareli,2025,379595,17.03790,14.57290,1.86235,271.63280,5.96326,1.45729,...,0.73409,9,Çok yüksek öncelik,0.01518,695.0,308.0,1242.0,1003.0,1043.0,10.0
9,Balıkesir,balikesir,2025,1284517,18.16698,15.22347,2.08055,275.08535,6.35844,1.52235,...,0.72941,10,Çok yüksek öncelik,0.01509,2859.0,1248.0,4930.0,3875.0,3453.0,25.0


In [36]:
# ============================================================
# 11) KAYDET
# ============================================================

scores_out.to_csv(
    "09_topsis_onem_siralamasi_2025_2030.csv",
    index=False,
    encoding="utf-8-sig"
)

weighted_out.to_csv(
    "09_topsis_agirlikli_kriterler_2025_2030.csv",
    index=False,
    encoding="utf-8-sig"
)

with pd.ExcelWriter("09_topsis_tek_dosya_2025_2030.xlsx") as writer:
    scores_out.to_excel(writer, sheet_name="TOPSIS_Onem_Siralamasi", index=False)
    weighted_out.to_excel(writer, sheet_name="Agirlikli_Kriterler", index=False)

    pd.DataFrame({
        "Kriter": list(WEIGHTS.keys()),
        "Ağırlık": list(WEIGHTS.values()),
        "Tip": [CRITERIA_TYPES[k] for k in WEIGHTS.keys()],
        "Yorum": [
            "65+ oranı arttıkça ihtiyaç artar",
            "0-14 oranı arttıkça ihtiyaç artar",
            "1000 kişi başına hekim azaldıkça ihtiyaç artar",
            "100 bin kişi başına yatak azaldıkça ihtiyaç artar",
        ]
    }).to_excel(writer, sheet_name="Kriter_Agirliklari", index=False)

print("Kaydedildi:")
print("09_topsis_onem_siralamasi_2025_2030.csv")
print("09_topsis_agirlikli_kriterler_2025_2030.csv")
print("09_topsis_tek_dosya_2025_2030.xlsx")
print()
print("Not: TOPSIS skoru yüksek olan il, ilgili yıl için daha yüksek ihtiyaç önceliğine sahiptir.")

Kaydedildi:
09_topsis_onem_siralamasi_2025_2030.csv
09_topsis_agirlikli_kriterler_2025_2030.csv
09_topsis_tek_dosya_2025_2030.xlsx

Not: TOPSIS skoru yüksek olan il, ilgili yıl için daha yüksek ihtiyaç önceliğine sahiptir.
